In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
# Step 1: Read credentials from secret scope
username = dbutils.secrets.get(scope="ecommerce_scope",key="azureSQL-username")
password = dbutils.secrets.get(scope="ecommerce_scope",key="azureSQL-password")

# Step 2: JDBC connection details
jdbc_hostname = "azuresqlserverkviswan8.database.windows.net"
jdbc_port = 1433
jdbc_database = "ecommerce-data-pipeline-db"

jdbc_url = f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};database={jdbc_database};encrypt=true;trustServerCertificate=false"

silver_transformations_rules = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "metadata_silver_config")  # schema.table
    .option("user", username)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

watermark_metadata = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "watermark_metadata")  # schema.table
    .option("user", username)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

metadata_df = silver_transformations_rules.join(watermark_metadata,on="table_name",how="inner")
display(metadata_df)

In [0]:
def getValidFilePathsofSilver(path,gold_last_processed):
    valid_paths = []
    folders = dbutils.fs.ls(path)
    
    for folder in folders:
        folder_name = folder.name.strip("/")
        folder_splits = folder_name.split("=")
        if(len(folder_splits)>1):
            date_part = folder_splits[1]
            folder_date = datetime.strptime(date_part, "%Y-%m-%d")

            if folder_date >= gold_last_processed and folder_date <= datetime.now():
                valid_paths.append(folder.path)
    return valid_paths

In [0]:
tables_dict = {}

for row in metadata_df.toLocalIterator():
    table_name = row['table_name']
    gold_watermark_column = row['gold_watermark_column']
    gold_last_processed = row['gold_last_processed']

    base_path = f"abfss://silverlayer@ecommercepipelineadls.dfs.core.windows.net/ecommerce_db/{table_name}"

    valid_paths = getValidFilePathsofSilver(base_path, gold_last_processed)

    if len(valid_paths) == 0:
        df = spark.read.format("delta").load(base_path)
    else:
        df = spark.read.format("delta").load(valid_paths)

    df = df.filter(col(gold_watermark_column) > lit(gold_last_processed))

    tables_dict[table_name] = df

# Ensure tables exist
required = ["customers","orders","order_items","products","payments"]

for t in required:
    if tables_dict.get(t) is None:
        schema = spark.read.format("delta").load(
            f"abfss://silverlayer@ecommercepipelineadls.dfs.core.windows.net/ecommerce_db/{t}"
        ).schema
        tables_dict[t] = spark.createDataFrame([], schema)

customers_df = tables_dict["customers"]
orders_df = tables_dict["orders"]
order_items_df = tables_dict["order_items"]
products_df = tables_dict["products"]
payments_df = tables_dict["payments"]

orders = orders_df.alias("o")
customers = customers_df.alias("c")
order_items = order_items_df.alias("oi")
products = products_df.alias("p")
payments = payments_df.alias("pay")

big_df = orders \
    .join(customers, col("o.customer_id") == col("c.customer_id"), "left") \
    .join(order_items, col("o.order_id") == col("oi.order_id"), "left") \
    .join(products, col("oi.product_id") == col("p.product_id"), "left") \
    .join(payments, col("o.order_id") == col("pay.order_id"), "left")

big_df = big_df \
    .withColumn("unit_price", col("oi.price")) \
    .withColumn("total_price", col("oi.price") * col("oi.quantity"))

In [0]:
#Helpers

gold_path = "abfss://goldlayer@ecommercepipelineadls.dfs.core.windows.net/ecommerce_db"

paths = {
    "dim_customers": f"{gold_path}/dim_customers",
    "dim_products": f"{gold_path}/dim_products",
    "dim_date": f"{gold_path}/dim_date",
    "dim_payments": f"{gold_path}/dim_payments",
    "dim_order_status": f"{gold_path}/dim_order_status",
    "fact_orders": f"{gold_path}/fact_orders"
}

def exists(path):
    return DeltaTable.isDeltaTable(spark, path)

In [0]:


dim_cust_src = big_df.select(
    col("c.customer_id"),
    col("c.first_name"),
    col("c.last_name"),
    col("c.email"),
    col("c.phone"),
    col("c.city"),
    col("c.full_name")
).dropDuplicates()

if not exists(paths["dim_customers"]):

    w = Window.orderBy("c.customer_id")

    dim_cust = dim_cust_src \
        .withColumn("surrogate_cus_id", row_number().over(w)) \
        .withColumn("effective_start_date", current_timestamp()) \
        .withColumn("effective_end_date", lit(None).cast("timestamp")) \
        .withColumn("is_current", lit(True))

    dim_cust.write.format("delta").mode("overwrite").save(paths["dim_customers"])

else:

    tgt = DeltaTable.forPath(spark, paths["dim_customers"])
    tgt_df = spark.read.format("delta").load(paths["dim_customers"])

    current_tgt = tgt_df.filter("is_current = true")

    changes = dim_cust_src.alias("src").join(
        current_tgt.alias("tgt"),
        "customer_id"
    ).filter("""
        tgt.first_name <> src.first_name OR
        tgt.last_name <> src.last_name OR
        tgt.email <> src.email OR
        tgt.city <> src.city OR
        tgt.phone_no <> src.phone_no
    """).select("src.*")

    staged = dim_cust_src.withColumn("merge_key", col("customer_id")) \
        .union(
            changes.withColumn("merge_key", lit(None))
        )

    max_key = tgt_df.agg(max("surrogate_cus_id")).collect()[0][0]
    max_key = max_key if max_key else 0

    w = Window.orderBy("customer_id")

    staged = staged \
        .withColumn("surrogate_cus_id", row_number().over(w) + max_key) \
        .withColumn("effective_start_date", current_timestamp()) \
        .withColumn("effective_end_date", lit(None).cast("timestamp")) \
        .withColumn("is_current", lit(True))

    tgt.alias("tgt").merge(
        staged.alias("src"),
        "tgt.customer_id = src.merge_key AND tgt.is_current = true"
    ).whenMatchedUpdate(
        condition="""
            tgt.first_name <> src.first_name OR
            tgt.last_name <> src.last_name OR
            tgt.email <> src.email OR
            tgt.city <> src.city OR
            tgt.phone_no <> src.phone_no
        """,
        set={
            "effective_end_date": "current_timestamp()",
            "is_current": "false"
        }
    ).whenNotMatchedInsert(
        values={
            "surrogate_cus_id": "src.surrogate_cus_id",
            "customer_id": "src.customer_id",
            "first_name": "src.first_name",
            "last_name": "src.last_name",
            "full_name": "src.full_name",
            "email": "src.email",
            "phone_no": "src.phone_no",
            "city": "src.city",
            "effective_start_date": "src.effective_start_date",
            "effective_end_date": "src.effective_end_date",
            "is_current": "src.is_current"
        }
    ).execute()

In [0]:

# SOURCE
dim_prod_src = big_df.select(
    col("p.product_id"),
    col("p.product_name"),
    col("p.category")
).dropDuplicates()


if not exists(paths["dim_products"]):

    w = Window.orderBy("p.product_id")

    dim_prod = dim_prod_src \
        .withColumn("surrogate_prod_id", row_number().over(w)) \
        .withColumn("effective_start_date", current_timestamp()) \
        .withColumn("effective_end_date", lit(None).cast("timestamp")) \
        .withColumn("is_current", lit(True))

    dim_prod.write.format("delta").mode("overwrite").save(paths["dim_products"])

else:

    tgt = DeltaTable.forPath(spark, paths["dim_products"])
    tgt_df = spark.read.format("delta").load(paths["dim_products"])
    current_tgt = tgt_df.filter("is_current = true")

    changes = dim_prod_src.alias("src").join(
        current_tgt.alias("tgt"),
        "product_id"
    ).filter("""
        tgt.product_name <> src.product_name OR
        tgt.category <> src.category
    """).select("src.*")

    staged = dim_prod_src.alias("src") \
        .withColumn("merge_key", col("product_id")) \
        .union(
            changes.withColumn("merge_key", lit(None))  # forces NOT MATCH
        )

    max_key = tgt_df.agg(max("surrogate_prod_id")).collect()[0][0]
    max_key = max_key if max_key else 0

    w = Window.orderBy("product_id")

    staged = staged \
        .withColumn("surrogate_prod_id", row_number().over(w) + max_key) \
        .withColumn("effective_start_date", current_timestamp()) \
        .withColumn("effective_end_date", lit(None).cast("timestamp")) \
        .withColumn("is_current", lit(True))

    tgt.alias("tgt").merge(
        staged.alias("src"),
        "tgt.product_id = src.merge_key AND tgt.is_current = true"
    ).whenMatchedUpdate(
        condition="""
            tgt.product_name <> src.product_name OR
            tgt.category <> src.category
        """,
        set={
            "effective_end_date": "current_timestamp()",
            "is_current": "false"
        }
    ).whenNotMatchedInsert(
        values={
            "surrogate_prod_id": "src.surrogate_prod_id",
            "product_id": "src.product_id",
            "product_name": "src.product_name",
            "category": "src.category",
            "effective_start_date": "src.effective_start_date",
            "effective_end_date": "src.effective_end_date",
            "is_current": "src.is_current"
        }
    ).execute()

In [0]:
#dim_date
dim_date_src = big_df.select(
    col("o.order_date").alias("date"),
    col("pay.payment_date")
).dropDuplicates()

dim_date_src = dim_date_src \
    .withColumn("year", year("date")) \
    .withColumn("month", month("date")) \
    .withColumn("quarter", quarter("date")) \
    .withColumn("day_of_week", dayofweek("date"))

if not exists(paths["dim_date"]):

    w = Window.orderBy("date")

    dim_date = dim_date_src \
        .withColumn("surrogate_date_id", row_number().over(w))

    dim_date.write.format("delta").mode("overwrite").save(paths["dim_date"])

else:

    tgt = DeltaTable.forPath(spark, paths["dim_date"])
    tgt_df = spark.read.format("delta").load(paths["dim_date"])

    max_key = tgt_df.agg(max("surrogate_date_id")).collect()[0][0]
    max_key = max_key if max_key else 0

    new_rows = dim_date_src.alias("src").join(
        tgt_df.alias("tgt"),
        "date",
        "left_anti"
    )

    w = Window.orderBy("date")

    new_rows = new_rows \
        .withColumn("surrogate_date_id", row_number().over(w) + max_key)

    tgt.alias("tgt").merge(
        dim_date_src.alias("src"),
        "tgt.date = src.date"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

In [0]:
#dim_payments

dim_payments_src = big_df.select(
    "pay.payment_id",
    "pay.payment_method",
    "pay.payment_status"
).dropDuplicates()

if not DeltaTable.isDeltaTable(spark, paths["dim_payments"]):
    w = Window.orderBy("payment_id")

    dim_payments = dim_payments_src.withColumn(
        "surrogate_payment_id", row_number().over(w)
    )

    dim_payments.write.format("delta").mode("overwrite").save(paths["dim_payments"])

else:
    tgt = DeltaTable.forPath(spark, paths["dim_payments"])

    tgt.alias("tgt").merge(
        dim_payments_src.alias("src"),
        "tgt.payment_id = src.payment_id"
    ).whenMatchedUpdate(
        set={
            "payment_method": "src.payment_method",
            "payment_status": "src.payment_status"
        }
    ).whenNotMatchedInsert(
        values={
            "payment_id": "src.payment_id",
            "payment_method": "src.payment_method",
            "payment_status": "src.payment_status"
        }
    ).execute()

In [0]:

dim_order_status_src = big_df.select(
    col("status")
).dropDuplicates()

if not DeltaTable.isDeltaTable(spark, paths["dim_order_status"]):

    w = Window.orderBy("status")

    dim_order_status = dim_order_status_src.withColumn(
        "surrogate_order_status_id", row_number().over(w)
    )

    dim_order_status.write.format("delta").mode("overwrite").save(paths["dim_order_status"])

else:

    tgt = DeltaTable.forPath(spark, paths["dim_order_status"])

    max_key = tgt.toDF().agg(max("surrogate_order_status_id")).collect()[0][0]
    max_key = max_key if max_key is not None else 0

    new_records = dim_order_status_src.alias("src") \
        .join(tgt.toDF().alias("tgt"), "status", "left_anti")

    w = Window.orderBy("status")

    new_records = new_records.withColumn(
        "surrogate_order_status_id",
        row_number().over(w) + max_key
    )

    new_records.write.format("delta").mode("append").save(paths["dim_order_status"])

In [0]:
dim_customers_df = spark.read.format("delta").load(paths["dim_customers"])
dim_products_df = spark.read.format("delta").load(paths["dim_products"])
dim_date_df = spark.read.format("delta").load(paths["dim_date"])
dim_payments_df = spark.read.format("delta").load(paths["dim_payments"])
dim_order_status_df = spark.read.format("delta").load(paths["dim_order_status"])

In [0]:
#Fact Table

fact_df = big_df \
    .join(dim_customers_df, "customer_id", "left") \
    .join(dim_products_df, "product_id", "left") \
    .join(dim_date_df.alias("od"), col("order_date") == col("od.date"), "left") \
    .join(dim_date_df.alias("pd"), col("pd.payment_date") == col("pd.date"), "left") \
    .join(dim_payments_df, "payment_id", "left") \
    .join(dim_order_status_df, "status", "left")

fact_final = fact_df.select(
    col("oi.order_item_id").alias("order_item_id"),
    col("o.order_id").alias("order_id"),
    col("surrogate_cus_id"),
    col("surrogate_prod_id"),
    col("od.surrogate_date_id").alias("order_date_id"),
    col("pd.surrogate_date_id").alias("payment_date_id"),
    col("surrogate_payment_id"),
    col("surrogate_order_status_id"),
    col("oi.quantity").alias("quantity"),
    col("unit_price"),
    col("total_price")
)

if not DeltaTable.isDeltaTable(spark, paths["fact_orders"]):

    fact_final.write.format("delta").mode("overwrite").save(paths["fact_orders"])

else:
    tgt = spark.read.format("delta").load(paths["fact_orders"])
    new_records = fact_final.alias("src").join(
        tgt.alias("tgt"),
        "order_item_id",
        "left_anti"
    )
    new_records.write.format("delta").mode("append").save(paths["fact_orders"])
